In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from shapely.geometry import box
import earthaccess as ea
from pathlib import Path
import rioxarray
import contextily as ctx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Image, display

In [3]:
data_dir = Path("./SWOT_IW_Labeled_Dataset/Andaman Sea/")

nc_files = sorted(data_dir.glob("data/SWOT_L2_LR_SSH_Expert_001_411_20230804T210804_20230804T215833_PGC0_01.nc"))
obb_label = sorted(data_dir.glob("obb_labels/SWOT_L2_LR_SSH_Expert_001_411_20230804T210804_20230804T215833_PGC0_01.txt"))
bb_label = sorted(data_dir.glob("bb_labels/SWOT_L2_LR_SSH_Expert_001_411_20230804T210804_20230804T215833_PGC0_01.txt"))

ds = xr.open_mfdataset(
                nc_files,
                combine="nested",
                decode_timedelta=True,
                compat='no_conflicts',
                engine='h5netcdf',
                parallel=True,
            )

ds

<xarray.Dataset> Size: 702kB
Dimensions:  (lat: 848, lon: 69)
Coordinates:
    lat      (lat, lon) float32 234kB dask.array<chunksize=(848, 69), meta=np.ndarray>
    lon      (lat, lon) float32 234kB dask.array<chunksize=(848, 69), meta=np.ndarray>
Data variables:
    ssha     (lat, lon) float32 234kB dask.array<chunksize=(848, 69), meta=np.ndarray>

In [4]:
with open(obb_label[0], "r", encoding="utf-8") as f:
    obb_text = f.read()

with open(bb_label[0], "r", encoding="utf-8") as f:
    bb_text = f.read()
print("OBB Labels:\n")
print(obb_text)
print("\nBB Labels:\n")
print(bb_text)

OBB Labels:

0 1.018860 0.044031 1.019242 0.002956 0.019243 0.002894 0.018860 0.043969
0 -0.054008 0.616277 0.312844 0.635817 0.559235 0.605189 0.192383 0.585650
0 -0.055747 0.545921 1.004704 0.546337 1.009520 0.464970 -0.050931 0.464554
0 0.020549 0.369503 0.451845 0.369153 0.442160 0.290113 0.010864 0.290463

BB Labels:

0 0.519051 0.023463 1.000382 0.041137
0 0.252614 0.610734 0.613243 0.050167
0 0.476886 0.505445 1.065267 0.081783
0 0.231354 0.329808 0.440981 0.079390


In [6]:
import numpy as np
import pandas as pd
import plotly.express as px

ds_plot = ds.isel(file=0) if "file" in ds.dims else ds

lat = ds_plot["lat"].values
lon = ds_plot["lon"].values
ssha = ds_plot["ssha"].values

df = pd.DataFrame({
    "lat": lat.ravel(),
    "lon": lon.ravel(),
    "ssha": ssha.ravel()
}).dropna(subset=["lat", "lon", "ssha"])

# Use only the central 95% color range
vmin, vmax = np.nanpercentile(df["ssha"], [2.5, 70])

fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    color="ssha",
    color_continuous_scale="RdBu_r",
    range_color=(vmin, vmax),
    zoom=4,
    height=700,
    opacity=0.8,
    title="Interactive SWOT SSHA Map"
)

fig.update_traces(marker=dict(size=4))

fig.update_layout(
    map_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

In [ ]:
import numpy as np
import xarray as xr
from sklearn.decomposition import PCA
from pyproj import Transformer

# Convert lat/lon to projected coordinates (meters)
transformer = Transformer.from_crs(
    "EPSG:4326", "EPSG:3857", always_xy=True
)

X, Y = transformer.transform(
    ds.lon.values,
    ds.lat.values
)

# Flatten points
pts = np.column_stack([X.ravel(), Y.ravel()])

# Find principal direction of swath
pca = PCA(n_components=2)
rot = pca.fit_transform(pts)

# Reshape back
y_rot = rot[:, 0].reshape(X.shape)  # along-track
x_rot = rot[:, 1].reshape(X.shape)  # across-track

# Create 1D coordinates
y = y_rot.mean(axis=1)
x = x_rot.mean(axis=0)

# Build new dataset
ds_xy = (
    ds.rename_dims({"lat": "y", "lon": "x"})
      .rename_vars({"lat": "latitude", "lon": "longitude"})
      .assign_coords(
          y=("y", y),
          x=("x", x)
      )
)

print(ds_xy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# -----------------------------
# Load OBBs (normalized)
# -----------------------------
def load_obb(file_path):
    boxes = []
    with open(file_path, "r") as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            coords = np.array(parts[1:]).reshape(4, 2)
            boxes.append(coords)
    return boxes

boxes = load_obb(obb_label[0])

# -----------------------------
# Image
# -----------------------------
ssha = ds_xy.ssha.values   # NO flip
ny, nx = ssha.shape

vmin, vmax = np.nanpercentile(ssha, [2.5, 80])

# -----------------------------
# Convert boxes → pixel space
# -----------------------------
def denormalize(coords, nx, ny):
    out = coords.copy()
    out[:, 0] *= nx
    out[:, 1] *= ny
    return out

# -----------------------------
# FLIP BOXES ONLY (vertical flip)
# -----------------------------
def flip_boxes_vertical(coords, ny):
    coords = coords.copy()
    coords[:, 1] = ny - coords[:, 1]
    return coords

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(4, 10))

im = ax.imshow(
    ssha,
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax,
    origin="lower",
    aspect="equal"
)

# -----------------------------
# Draw flipped boxes
# -----------------------------
for coords in boxes:
    coords_px = denormalize(coords, nx, ny)
    coords_px = flip_boxes_vertical(coords_px, ny)

    ax.add_patch(
        Polygon(
            coords_px,
            closed=True,
            edgecolor="yellow",
            facecolor="none",
            linewidth=2
        )
    )

ax.set_xlim(0, nx)
ax.set_ylim(0, ny)

ax.set_title("SWOT SSHA + vertically flipped OBBs")
plt.colorbar(im, ax=ax)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# -----------------------------
# Load YOLO BB labels
# -----------------------------
def load_yolo_bb(file_path):
    boxes = []
    with open(file_path, "r") as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            cls, xc, yc, w, h = parts
            boxes.append((cls, xc, yc, w, h))
    return boxes

boxes = load_yolo_bb(bb_label[0])

# -----------------------------
# Image
# -----------------------------
ssha = ds_xy.ssha.values
ny, nx = ssha.shape

vmin, vmax = np.nanpercentile(ssha, [2.5, 80])

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(4, 10))

ax.imshow(
    ssha,
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax,
    origin="lower",
    aspect="equal"
)

# -----------------------------
# Draw flipped YOLO boxes
# -----------------------------
for cls, xc, yc, w, h in boxes:

    # ✔️ FLIP Y FIRST (normalized space)
    yc = 1.0 - yc

    # convert to pixel space
    xc *= nx
    yc *= ny
    w *= nx
    h *= ny

    xmin = xc - w / 2
    ymin = yc - h / 2

    rect = Rectangle(
        (xmin, ymin),
        w,
        h,
        edgecolor="yellow",
        facecolor="none",
        linewidth=2
    )
    ax.add_patch(rect)

ax.set_xlim(0, nx)
ax.set_ylim(0, ny)

ax.set_title("SWOT SSHA + flipped YOLO BBs")
plt.colorbar(ax.images[0], ax=ax)
plt.show()